In [1]:
import pandas as pd
import re
import json
import emoji
from collections import Counter
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from mistralai.client import Mistral
import os
import unicodedata

tqdm.pandas()

KeyboardInterrupt: 

In [ ]:
def extract_systematic_oov(df, text_column="text_norm"):
    print("⏳ Đang trích xuất từ lạ bằng Soft Filter (Ngưỡng động)...")
    try:
        with open("Viet74K.txt", "r", encoding="utf-8") as f:
            standard_dict = set(f.read().splitlines())
    except FileNotFoundError:
        standard_dict = set(["tôi", "bạn", "là", "đi", "làm"])

    unknown_counter = Counter()
    
    for text in df[text_column].dropna():
        # Xóa dấu câu, chuyển chữ thường
        clean_text = re.sub(r'[^\w\s]', '', str(text).lower())
        words = clean_text.split()
        
        for word in words:
            if word.isdigit():
                continue
                
            # KỸ THUẬT 1: Chuẩn hóa ký tự kéo dài (ví dụ: koooo -> ko, đẹpppp -> đẹp)
            # Regex này gom 3 ký tự giống nhau liên tiếp trở lên thành 1 ký tự
            normalized_word = re.sub(r'([a-zA-Z])\1{2,}', r'\1', word)
            
            if normalized_word not in standard_dict:
                unknown_counter[normalized_word] += 1
                
    systematic_oov = {}
    for word, count in unknown_counter.items():
        word_len = len(word)
        
        if word_len <= 3 and count >= 3:
            systematic_oov[word] = count
            
        elif word_len > 3 and count >= 5:
            systematic_oov[word] = count
            
    # Sắp xếp theo tần suất giảm dần
    sorted_oov = sorted(systematic_oov.items(), key=lambda x: x[1], reverse=True)
    
    print(f"✅ Đã lọc được {len(sorted_oov)} từ lạ tiềm năng.")
    return sorted_oov

In [ ]:
def generate_teencode_dict_with_mistral(oov_list, api_key, model_name="mistral-large-latest", output_file="custom_teencode.json"):
    print(f"⏳ Đang gửi danh sách OOV cho mô hình {model_name}...")
    
    client = Mistral(api_key=api_key)
    
    # Chuẩn bị Prompt
    prompt = """Bạn là một chuyên gia ngôn ngữ mạng xã hội Việt Nam. 
Dưới đây là danh sách các từ lạ (kèm số lần xuất hiện) trích xuất từ dữ liệu.
Hãy tạo một file JSON (`"từ lóng": "từ chuẩn"`) với 3 quy tắc NGHIÊM NGẶT sau:
1. CHỈ map các từ lóng (teencode), từ viết tắt cố ý (VD: 'ko' -> 'không', 'đz' -> 'đẹp trai', 'chs' -> 'chơi', 'cmm' -> 'con mẹ mày', cmn -> 'con mẹ nó').
2. TUYỆT ĐỐI BỎ QUA các lỗi sai chính tả vô ý (VD: 'dẽ', 'qúa').
3. TUYỆT ĐỐI BỎ QUA các tên riêng, tiếng nước ngoài (VD: 'picasso', 'idol').
4. dịch được ít nhất 50 từ lóng phổ biến nhất trong danh sách dưới đây (từ có tần suất cao nhất).
Dữ liệu (Top từ lạ):
"""
    for word, count in oov_list[:250]:
        prompt += f"- {word}: {count}\n"
    
    prompt += "\nHãy chỉ trả về mã JSON, tuyệt đối không có văn bản giải thích nào khác."

    # Gọi API
    try:
        response = client.chat.complete(
            model=model_name,
            messages=[
                {"role": "system", "content": "Bạn là hệ thống chuyển đổi dữ liệu sang định dạng JSON nguyên chuẩn, không bọc thêm bất kỳ markdown nào nếu không cần thiết."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3 # Giữ temperature cực thấp để tránh hallucination
        )
        
        llm_output = response.choices[0].message.content
        print("✅ Đã nhận phản hồi từ Mistral API.")
        
        # Bóc tách JSON an toàn (LLM thường hay bọc trong ```json ... ```)
        json_pattern = re.search(r'\{.*\}', llm_output, re.DOTALL)
        if json_pattern:
            json_str = json_pattern.group(0)
            
            # Kiểm tra xem chuỗi sinh ra có phải JSON hợp lệ không
            parsed_json = json.loads(json_str)
            
            # Lưu ra file
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(parsed_json, f, ensure_ascii=False, indent=4)
            print(f"✅ Đã lưu từ điển tự động vào file: {output_file}")
            print(f"✅ Đã trích xuất được {len(parsed_json)} quy tắc chuẩn hóa.")
            
        else:
            print("❌ Lỗi: Không tìm thấy định dạng JSON trong phản hồi của LLM.")
            print("Phản hồi gốc:\n", llm_output)

    except Exception as e:
        print(f"❌ Có lỗi xảy ra khi gọi Mistral API: {e}")

In [ ]:
class TextNormalizerPipeline:
    def __init__(self, dict_path="master_teencode.json", model_name="yammdd/vietnamese-error-correction"):
        # 1. KHỞI TẠO TỪ ĐIỂN VÀ BIÊN DỊCH REGEX

        try:
            with open(dict_path, "r", encoding="utf-8") as f:
                raw_dict = json.load(f)
                
            # Đồng bộ Unicode NFC và chuyển chữ thường để tránh lỗi bảng mã
            self.teencode_dict = {
                unicodedata.normalize('NFC', k.lower()): v 
                for k, v in raw_dict.items()
            }
            
            # Sắp xếp theo độ dài giảm dần để tránh lỗi đè chữ (overlapping)
            sorted_keys = sorted(self.teencode_dict.keys(), key=len, reverse=True)
            escaped_keys = [re.escape(k) for k in sorted_keys]
            
            # Biên dịch thành 1 mẫu Regex duy nhất để quét siêu tốc
            self.pattern = re.compile(rf"\b({'|'.join(escaped_keys)})\b", re.IGNORECASE)
            print(f"✅ Đã nạp và biên dịch xong {len(sorted_keys)} luật Teencode từ {dict_path}.")
            
        except FileNotFoundError:
            self.teencode_dict = {}
            self.pattern = None
            print(f"⚠️ CẢNH BÁO: Không tìm thấy file {dict_path}.")


        # 2. TẢI MÔ HÌNH HUGGING FACE SỬA TYPO

        print("⏳ Đang tải AI sửa lỗi chính tả (BARTpho)...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(self.device)
        print(f"✅ Đã tải xong AI trên {self.device.upper()}.")

        # Danh sách các từ tiếng Anh hoặc tên riêng không chứa f, j, w, z nhưng vẫn muốn bảo vệ
        self.protected_words = ['boy', 'hot', 'trend', 'idol', 'fan', 'hit', 'view', 'nick', 'minaj', 'picasso']

    def process_text(self, text):
        if not isinstance(text, str) or text.strip() == "":
            return text
            

        # BƯỚC 1: TIỀN XỬ LÝ & RULE-BASED (LƯỚI 1)

        # Đồng bộ Unicode và thu gọn ký tự kéo dài (koooo -> ko)
        text = unicodedata.normalize('NFC', text)
        text = re.sub(r'([a-zA-Z])\1{2,}', r'\1', text)
        
        # Chuyển Emoji thành dạng text (vd: :face_with_tears_of_joy:)
        text = emoji.demojize(text)
        
        # Map Teencode (Tra từ điển bằng Regex)
        if self.pattern:
            text = self.pattern.sub(
                lambda m: self.teencode_dict.get(m.group(1).lower(), m.group(0)), 
                text
            )


        # BƯỚC 2: MASKING (CHE MẶT NẠ BẢO VỆ DỮ LIỆU)

        mask_dict = {}
        mask_counter = 0

        # Hàm trợ giúp để gỡ từ ra và thế token [MASK] vào
        def mask_replacer(match):
            nonlocal mask_counter
            mask_token = f" MASKTOKEN{mask_counter} "
            mask_dict[mask_token.strip()] = match.group(0)
            mask_counter += 1
            return mask_token

        # 2.1 Che Emoji đã demojize
        text = re.sub(r':[a-zA-Z0-9_]+:', mask_replacer, text)
        
        # 2.2 Che tiếng Anh chứa ký tự ngoại lai (f, j, w, z) - Vd: fuck, minaj
        text = re.sub(r'\b[a-zA-Z]*[fjwzFJWZ][a-zA-Z]*\b', mask_replacer, text)
        
        # 2.3 Che các từ tiếng Anh phổ biến tự định nghĩa ở trên
        for p_word in self.protected_words:
            text = re.sub(rf'\b{p_word}\b', mask_replacer, text, flags=re.IGNORECASE)

        # BƯỚC 3: INFER MODEL SỬA TYPO (LƯỚI 2)

        clean_text = " ".join(text.split())
        if not clean_text: return text
        
        # Đưa câu chứa MASK qua mô hình AI
        inputs = self.tokenizer(clean_text, return_tensors="pt", max_length=256, truncation=True).to(self.device)
        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_length=256)
            
        corrected_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)


        # BƯỚC 4: UNMASKING (KHÔI PHỤC DỮ LIỆU GỐC)

        for mask_token, original_word in mask_dict.items():
            corrected_text = corrected_text.replace(mask_token, original_word)
            
        return " ".join(corrected_text.split())

In [ ]:
def merge_teencode_dictionaries():
    master_dict = {}

    # 1. Nạp từ điển nền (teencode.txt từ GitHub)
    # File txt thường có cấu trúc: từ_lóng \t từ_chuẩn
    if os.path.exists("teencode.txt"):
        with open("teencode.txt", "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 2:
                    master_dict[parts[0].strip()] = parts[1].strip()
        print(f"Đã nạp file txt. Số lượng từ hiện tại: {len(master_dict)}")

    # 2. Nạp và GHI ĐÈ bằng 4 file JSON của bạn (Độ ưu tiên cao hơn)
    json_files = [
        "custom_teencode_1.json", 
        "custom_teencode_2.json", 
        "custom_teencode_3.json", 
        "custom_teencode_4.json",
        "custom_teencode.json"
    ]
    
    for file in json_files:
        if os.path.exists(file):
            with open(file, "r", encoding="utf-8") as f:
                data = json.load(f)
                master_dict.update(data) # Lệnh update sẽ tự động ghi đè nếu trùng key
                print(f"Đã nạp {file}. Tổng số từ: {len(master_dict)}")

    # 3. Lưu thành file tổng chuẩn nhất
    with open("master_teencode.json", "w", encoding="utf-8") as f:
        json.dump(master_dict, f, ensure_ascii=False, indent=4)
        
    print(f"\n✅ ĐÃ TẠO MASTER DICTIONARY THÀNH CÔNG: {len(master_dict)} TỪ!")
    return master_dict

In [ ]:
def get_remaining_oov(original_oov_list, master_dict):
    remaining_oov = []
    
    for word, count in original_oov_list:
        # Nếu từ lạ chưa tồn tại trong từ điển tổng, ta giữ lại để mang đi hỏi LLM
        if word not in master_dict:
            remaining_oov.append((word, count))
            
    print(f"\nDanh sách OOV gốc: {len(original_oov_list)} từ")
    print(f"Danh sách OOV còn lại cần dịch: {len(remaining_oov)} từ")
    return remaining_oov

In [ ]:
if __name__ == "__main__":
    # Đọc dữ liệu
    df = pd.read_csv("/home/check/DATA/university/yr3, hk2/IE403 - Khai thác dữ liệu truyền thông xã hội/Data/bronze/final_commonsense_reverted.csv")
    
    # MISTRAL_API_KEY = ""
    # oov_list = extract_systematic_oov(df, "text_norm")
    # master_dict = merge_teencode_dictionaries()
    # remaining_oov = get_remaining_oov(oov_list, master_dict)
    # generate_teencode_dict_with_mistral(remaining_oov, api_key=MISTRAL_API_KEY, output_file="teencode_combined_1.json")

    # xóa timestamp và khoảng trắng thừa
    df['text_norm'] = df['text_norm'].astype(str).str.replace(
    r'\b\d{1,2}[:\s]\d{2}(?::\d{2})?\b', 
    '', 
    regex=True)
    df['text_norm'] = df['text_norm'].str.replace(r'\s+', ' ', regex=True).str.strip()
    # # Khởi tạo Pipeline
    pipeline = TextNormalizerPipeline(dict_path="teencode_data/master_teencode.json")
    
    # Chạy trên 10 dòng đầu tiên để test 
    sample_df = df.head(10).copy()
    print("⏳ Đang chuẩn hóa dữ liệu...")
    
    # # progress_apply giúp hiện thanh tiến trình % thay vì chờ màn hình đen
    sample_df['text_fully_cleaned'] = sample_df['text_norm'].progress_apply(pipeline.process_text)
    
    # Xem thử kết quả
    print("\nKết quả Test:")
    for idx, row in sample_df.iterrows():
        print(f"Gốc : {row['text_norm']}")
        print(f"Sạch: {row['text_fully_cleaned']}")
        print("-" * 30)

    df['text_fully_cleaned'] = df['text_norm'].progress_apply(pipeline.process_text)
    df.to_csv("comment_cleaned.csv", index=False)

✅ Đã nạp và biên dịch xong 576 luật Teencode từ teencode_data/master_teencode.json.
⏳ Đang tải AI sửa lỗi chính tả (BARTpho)...
✅ Đã tải xong AI trên CUDA.
⏳ Đang chuẩn hóa dữ liệu...


100%|██████████| 10/10 [00:01<00:00,  8.63it/s]



Kết quả Test:
Gốc : bác pi cát xô là bạn của bác hồ
Sạch: bác pi cát xô là bạn của bác hồ
------------------------------
Gốc : cấm sừng nhiều thế
Sạch: cấm sừng nhiều thế
------------------------------
Gốc : cho ai chưa biết thì những người đó đều là nicki minaj nhé
Sạch: cho ai chưa biết thì những người đó đều là nicki minaj nhé
------------------------------
Gốc : tên bài nhạc là gì vậy mn
Sạch: tên bài nhạc là gì vậy mọi người
------------------------------
Gốc : picasso năm 10 14t đz vải
Sạch: picasso năm 10 14t đẹp trai vãi
------------------------------
Gốc : đọc cái tên kiểu gì thế
Sạch: đọc cái tên kiểu gì thế
------------------------------
Gốc : oát đờ cái tên khai sinh
Sạch: oát đờ cái tên khai sinh
------------------------------
Gốc : fuck boy nhưng là hoạ sĩ
Sạch: fuck boy nhưng là hoạ sĩ
------------------------------
Gốc : pacasso chúa tể cắm sừng
Sạch: pacasso chúa tể cắm sừng
------------------------------
Gốc : chúa tể cắm sừng à 😂
Sạch: chúa tể cắm sừng à :face_with_

100%|██████████| 8122/8122 [22:03<00:00,  6.14it/s]  


In [ ]:

df = pd.read_csv("comment_cleaned.csv")
df['text_norm'] = df['text_fully_cleaned']
df['comment'] = df['text_norm']
df.drop(columns=['text_fully_cleaned','text_norm','commonsense'], inplace=True)
df = df[~df['comment'].astype(str).str.contains('sticker', case=False, na=False)]

df.to_csv("comment_cleaned_revised.csv", index=False, encoding="utf-8-sig")